# **Machine Learning with PyTorch**

PyTorch is an open source framework for AI research and commercial production in machine learning. It is used to build, train, and optimize deep learning neural networks for applications such as image recognition, natural language processing, and speech recognition. It provides computation support for CPU, GPU, parallel and distributed training on multiple GPUs and multiple nodes. PyTorch is also flexible and easily extensible, with specific libraries and tools available for many different domains. More information at https://pytorch.org

---


# Objectives

This notebook will be able to:

 - Install necessary PyTorch libraries;
 - Use PyTorch to build, train and evaluate neural networks.
 - Save the trained model parameters and use them later for inferencing.


---


# Setup

### Installing libraries

The following libraries are installed in Jupyter environment (e.g. VS Code, Sagemaker, Google Collab, or Ananconda), you will need to install these libraries.

*   [`torch`](https://pytorch.org/docs/stable/library.html?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkGuidedProjectsIBMDeveloperGPXX0W98EN3615-2023-01-01)
*   [`TorchVision`](https://pytorch.org/vision/stable/index.html?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkGuidedProjectsIBMDeveloperGPXX0W98EN3615-2023-01-01)


In [ ]:
!pip install torch torchvision matplotlib

### Importing libraries

I recommend import all required libraries in one place (here):_


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import numpy as np
import matplotlib.pyplot as plt

# How does this code work?
 
This notebook uses [MNIST dataset of Handwritten Digits](https://archive.ics.uci.edu/dataset/683/mnist+database+of+handwritten+digits). The dataset has over 60,000 images of hand written digits. The data will be partitioned between training the AI model and testing the AI model after training.

The main steps in this project include:
 1. Download the MNIST dataset and create a DataLoader for the dataset.
 2. Define an AI model to recognize a hand written digit.
 3. Train the defined AI model using training data from the MNIST dataset.
 4. Test the trained AI model using testing data from the MNIST dataset.
 5. Evaluate


# Download dataset and dreate Data Loader

The images are 28x28 pixel images of digits 0 through 9.


In [ ]:
# Download training data from MNIST datasets.
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

batch_size = 64

# Create data loaders to iterate over data
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

print("Training data size:", len(train_dataloader) * batch_size)
print("Test data size:", len(test_dataloader) * batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

In [ ]:
training_data.targets.numpy(), training_data.targets.numpy().shape, training_data.targets.numpy().dtype

In [ ]:
#print unique labels from training data
unique_labels = np.unique(training_data.targets.numpy(), return_counts=True)
unique_labels

### Display few images using matplotlib

In [ ]:
def display_few_images(sample_images, sample_labels):
    plt.figure(figsize=(10, 10))
    for i in range(25):
        idx = np.random.randint(0, len(sample_images))
        img = sample_images[idx]
        label = sample_labels[idx]
        plt.subplot(5, 5, i+1)
        plt.axis('off')
        plt.title(f"L: {label}")
        plt.imshow(img)
    plt.show()

display_few_images(training_data.data.numpy()[:10], training_data.targets.numpy()[:10])

# Define Model

I 1st determine the best device for performing training with cpu as the default device and then define the AI model as a neural network with 3 layers: an input layer, a hidden layer, and an output layer. Between the layers, we use a ReLU activation function.

Since the input images are 1x28x28 tensors, I need to flatten the input tensors into a 784 element tensor using the Flatten module before passing the input into our neural network.


In [ ]:
# Get device for training.
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() # Apple Silicon GPU
    else "cpu"
)
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, image_tensor):
        image_tensor = self.flatten(image_tensor)
        logits = self.linear_relu_stack(image_tensor)
        return logits

input_size = 28*28
hidden_size = 512
num_classes = 10

model = NeuralNetwork(input_size, hidden_size, num_classes).to(device)
print(model)

# Training loop

I implement a training function to use with the train_dataloader to train our model. Each iteration over the dataloader returns a batch_size image data tensor along with the expected output. After moving the tensors to the device, calling the forward pass of the model, compute the prediction error using the expected output and then call the backwards pass to compute the gradients and apply them to the model parameters.


In [ ]:
# Define our learning rate, loss function and optimizer
learning_rate = 1e-3 # 0.001
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Let's define our training function 
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()

    for batch_num, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Forward pass to compute prediction
        pred = model(X)
        # Compute prediction error using loss function
        loss = loss_fn(pred, y)

        # Backward pass
        optimizer.zero_grad() # zero any previous gradient calculations
        loss.backward() # calculate gradient
        optimizer.step() # update model parameters
        
        if batch_num > 0 and batch_num % 100 == 0:
            loss, current = loss.item(), batch_num * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

# Test Loop

The test methods evaluates the model's predictive performance using the test_dataloader. During testing, not require the gradient computation, so set the model in evaluate mode.


In [ ]:
# Our test function
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        test_loss += loss_fn(pred, y).item()
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

# Train the Model

Now, defined methods to train our model and test the trained model's predictive behavior, lets train the model for 10 epochs over the dataset.


In [ ]:
# Let's run training
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

# Save the model and make predictions

Once trained model, save the model parameters for future use in inferences. Save the state_dict of the model which contains the trained parameters. I then create a new instance of the model and load the previously saved parameters into the new instance of the model. Finally I can inference using the new instance of the model.


In [ ]:
# Save our model parameters
torch.save(model.state_dict(), "ml_with_pytorch_model.pth")
print("Saved PyTorch Model State to ml_with_pytorch_model.pth")

# Load the saved model parameters into a new instance of the model
model = NeuralNetwork(input_size, hidden_size, num_classes).to(device)
model.load_state_dict(torch.load("ml_with_pytorch_model.pth"))

# Inference using the new model instance
model.eval()
for i in range(10):
    x, y = test_data[i][0], test_data[i][1]

    x = x.to(device)
    pred = model(x)
    predicted, actual = pred[0].argmax(0).item(), y
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

# Congratulations! model prediction is completed
